# TCB: all review months combined

All 14 available review cycles (July 2024 to August 2026) are pooled here: **26,302 eligible reviews**, including **1,808 active/dying reviews**. Each item-stockroom-cycle record has equal weight. This includes the previously opened August test; it is a descriptive historical comparison, not a new independent validation.

The guarded wider-anchor candidate matches 993 active/dying reviews (54.92%) versus the current engine's 984 (54.42%), and reduces large underestimates from 275 to 262. Its earlier August result remains 144/226 versus 146/226. No production data or settings were changed.

## Data and calculation

Uses the same reconciled S27 panel and frozen engine/configuration. The 59 unresolved/conflicting/invalid records remain excluded. Legitimate records for the same part in different review cycles remain separate. Each prior decision still comes from an earlier cycle and the same stockroom. Missing cycles are not filled with zero demand.

Pooled agreement = total matching reviews / total reviews. A match requires both Max and ROP within 10% of the engineer's values; a zero target requires zero. Percentages are not an unweighted average of monthly percentages. Large underestimates use the same S27 definition: either Max or ROP below the engineer by more than max(1 unit, 25% of that target).


In [ ]:
from pathlib import Path
import json, hashlib
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p/'analysis/s28_pooled_months.py').exists())
OUT = ROOT/'analysis/output/s28_pooled_months'
SOURCE = ROOT/'analysis/output/s27_chronological'
summary = json.loads((OUT/'summary.json').read_text())
combined = pd.read_csv(OUT/'all_months_combined.csv')
monthly = pd.read_csv(OUT/'all_months_by_cycle.csv')
data = pd.read_csv(OUT/'combined_tcb_reviews.csv', dtype={'item_id':str, 'stockroom_id':str})
assert len(data) == summary['eligible_reviews'] == 26302
assert data['_cycle'].nunique() == 14
assert not data.duplicated(['item_id','stockroom_id','_cycle']).any()
for name, expected in summary['original_evidence_hashes'].items():
    assert hashlib.sha256((SOURCE/name).read_bytes()).hexdigest() == expected
names = {'single_flat_prior1':'Current engine',
         'single_flat_prior2_positive':'Guarded wider anchor',
         'single_flat_guard25_2':'Wider anchor + demand guard',
         'hold_current':'Hold current', 'last_approved':'Last approved'}
selected = combined[combined.candidate.isin(names)].copy()
table = selected[selected.segment.isin(['all','live'])].pivot(index='candidate', columns='segment', values='match_pct').reindex(names)
table.index = [names[n] for n in table.index]
table.columns = ['All reviews (%)', 'Active/dying (%)']
display(table.round(2))


## Combined agreement

The all-row figures are dominated by 24,494 dormant reviews. The active/dying column better distinguishes changes to demand-based sizing. Holding current levels has the highest pooled agreement but fails the error checks: it produces more incorrect zero Max/ROP values and reduces levels for critical parts relative to the engine.

The highest-agreement engine variant (wider anchor plus a demand guard) also fails the error checks. The positive-level guarded candidate retains a small pooled gain. None of these pooled comparisons supersedes the independent August evaluation.


In [ ]:
plt.rcParams.update({'font.size':10, 'figure.dpi':125, 'axes.spines.top':False, 'axes.spines.right':False})
colors=['#2364aa','#c35b16','#7d54a6','#66717e','#3d8270']
fig, axes = plt.subplots(1,2,figsize=(12.2,4.2),sharey=True)
for ax,column,title in zip(axes,table.columns,['All 26,302 reviews','1,808 active/dying reviews']):
    ax.barh(range(len(table)),table[column],color=colors,height=.6)
    ax.set_yticks(range(len(table)),table.index)
    ax.set(xlim=(0,105),xlabel='Max + ROP agreement (%)',title=title)
    ax.set_xticks([0,20,40,60,80,100])
    for i,value in enumerate(table[column]):
        ax.text(value+1,i,f'{value:.2f}%',va='center',fontsize=9)
axes[0].invert_yaxis()
fig.suptitle('All 14 review cycles combined - historical comparison',fontsize=14)
fig.tight_layout()
fig.savefig(OUT/'combined_comparison.png')
plt.show()
live = selected[selected.segment.eq('live')].set_index('candidate').reindex(names)
display(live[['n','matches','match_pct','severe_under','zero_max_errors','zero_rop_errors','critical_reductions_vs_baseline']].rename(index=names).round(2))


## Month-by-month reconciliation

Every candidate has the same monthly denominator. Total matches and weighted error metrics are reconciled to the original monthly scorecards. S28 also independently rescored the whole panel for the current engine, the prior selected candidate, the pooled leaders and the two simple comparators.


In [ ]:
for row in combined.to_dict('records'):
    scope=monthly[monthly.candidate.eq(row['candidate']) & monthly.segment.eq(row['segment'])]
    assert int(scope.n.sum()) == row['n']
    assert int(scope.matches.sum()) == row['matches']
    assert abs(100*scope.matches.sum()/scope.n.sum()-row['match_pct']) < 1e-10
by_cycle=monthly[monthly.segment.eq('live') & monthly.candidate.isin(['single_flat_prior1','single_flat_prior2_positive'])]
by_cycle=by_cycle.pivot(index='cycle',columns='candidate',values=['n','matches','match_pct'])
display(by_cycle.round(2))
print('Pooled totals reconciled; historical keys unique; earlier test evidence unchanged.')


## Files and reproduction

- `combined_tcb_reviews.csv`: all 26,302 eligible item-stockroom-cycle records with cycle labels and earlier-decision features.
- `all_months_combined.csv`: pooled metrics for 31 engine configurations plus two comparators, by segment.
- `all_months_by_cycle.csv`: underlying monthly metrics.
- `pooled_live_ranking.csv`: descriptive active/dying ranking and pooled error checks.
- `summary.json`: provenance, source hashes and paired uncertainty checks.

Run `python analysis/s28_pooled_months.py` from the project environment to reproduce the combined files. It reads frozen S27 evidence and writes only S28 local outputs. The ranking uses data already seen during development and final testing; it is not evidence of performance on a future cycle. Bootstrap intervals resample item-stockroom groups but do not correct for selecting configurations on these records. These agreement metrics do not measure actual stockouts or inventory savings.
